In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [4]:
import sqlite3
con = sqlite3.connect("../db.sqlite3")
cur = con.cursor()
res = cur.execute(
    "SELECT restaurant_order.id, datetime, name, quantity, price, quantity * price "
    "FROM restaurant_order "
    "JOIN restaurant_orderitem "
    "ON restaurant_order.id = restaurant_orderitem.order_id "
    "JOIN restaurant_product "
    "ON restaurant_orderitem.product_id = restaurant_product.id"
)

df = pd.DataFrame(res, columns=["order_id", "datetime", "product_name", "quantity", "price", "total_price"])
df.to_csv("restaurant_order_data.csv", index=False)
cur.close()

df = pd.read_csv("restaurant_order_data.csv")

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_10_by_qty = df.groupby("product_name")["quantity"].sum().sort_values(ascending=False)[:10]
fig, ax = plt.subplots(figsize=(10, 8))
ax.pie(
    top_10_by_qty,
    labels=top_10_by_qty.index,
    startangle=90,
    autopct=lambda pct: f"{pct:.1f}% ({int(pct/100 * top_10_by_qty.sum())})")
plt.title("Top 10 positions in menu by quantity")

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
top_10_by_total_price = df.groupby("product_name")["total_price"].sum().sort_values(ascending=False)[:10]
fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    top_10_by_total_price,
    labels=top_10_by_total_price.index,
    startangle=90,
    autopct=lambda pct: f"{pct:.1f}% ({int(pct/100 * top_10_by_total_price.sum())})")
plt.title("Top 10 positions make the most money for restaurant")

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_hour"] = pd.to_datetime(df["datetime"]).dt.hour
hourly_income = df.groupby("order_hour")["total_price"].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.bar(hourly_income["order_hour"], hourly_income["total_price"], color="skyblue", edgecolor="navy")
plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.xticks(range(24))
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["day_of_week"] = df["datetime"].dt.dayofweek
days = {0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"}
df["day_name"] = df["day_of_week"].map(days)
daily_income = df.groupby(["day_of_week", "day_name"])["total_price"].sum().reset_index().sort_values("day_of_week")

plt.figure(figsize=(12, 6))
plt.bar(daily_income["day_name"], daily_income["total_price"], color="skyblue", edgecolor="navy")
plt.title("Profit by Day of Week")
plt.xlabel("Day of Week")
plt.grid(axis="y", linestyle="--", alpha=0.7)